# Tugas Klasifikasi Naive Bayes Manual

Notebook ini dibuat untuk mensimulasikan langkah-langkah perhitungan Naive Bayes secara manual tanpa library seperti scikit-learn, agar proses perhitungan transparan sesuai dengan teori dasar Naive Bayes.

In [ ]:
# Data Latih List of Dictionaries
data_latih = [
    {'Harga': 'Murah', 'Diskon': 'Ada', 'Ulasan': 'Baik', 'Hasil': 'Tertarik'},
    {'Harga': 'Murah', 'Diskon': 'Ada', 'Ulasan': 'Baik', 'Hasil': 'Tertarik'},
    {'Harga': 'Sedang', 'Diskon': 'Ada', 'Ulasan': 'Baik', 'Hasil': 'Tertarik'},
    {'Harga': 'Mahal', 'Diskon': 'Tidak Ada', 'Ulasan': 'Buruk', 'Hasil': 'Tidak Tertarik'},
    {'Harga': 'Mahal', 'Diskon': 'Tidak Ada', 'Ulasan': 'Buruk', 'Hasil': 'Tidak Tertarik'},
    {'Harga': 'Sedang', 'Diskon': 'Ada', 'Ulasan': 'Baik', 'Hasil': 'Tertarik'},
    {'Harga': 'Sedang', 'Diskon': 'Tidak Ada', 'Ulasan': 'Baik', 'Hasil': 'Tertarik'},
    {'Harga': 'Mahal', 'Diskon': 'Ada', 'Ulasan': 'Buruk', 'Hasil': 'Tidak Tertarik'}
]

# Data Baru yang Diprediksi
data_baru = {
    'Harga': 'Sedang',
    'Diskon': 'Ada',
    'Ulasan': 'Baik'
}

print("Total data latih:", len(data_latih))
print("Data baru yang diprediksi:", data_baru)

## Langkah 1 & 2: Menentukan Kelas dan Prior Probability
Prior probability adalah probabilitas kemunculan masing-masing kelas dari total keseluruhan data latih.

Rumus: 
`P(Kelas) = Jumlah kemunculan Kelas / Total Data`

In [ ]:
total_data = len(data_latih)

# Menghitung jumlah masing-masing kelas
jumlah_tertarik = sum(1 for d in data_latih if d['Hasil'] == 'Tertarik')
jumlah_tidak_tertarik = sum(1 for d in data_latih if d['Hasil'] == 'Tidak Tertarik')

# Menghitung probabilitas prior
prior_tertarik = jumlah_tertarik / total_data
prior_tidak_tertarik = jumlah_tidak_tertarik / total_data

print(f"Jumlah data kelas 'Tertarik' = {jumlah_tertarik}")
print(f"Jumlah data kelas 'Tidak Tertarik' = {jumlah_tidak_tertarik}")
print(f"\nPrior P(Tertarik) = {jumlah_tertarik} / {total_data} = {prior_tertarik:.4f}")
print(f"Prior P(Tidak Tertarik) = {jumlah_tidak_tertarik} / {total_data} = {prior_tidak_tertarik:.4f}")

## Langkah 3: Menghitung Likelihood (dengan Laplace Smoothing)
Likelihood adalah probabilitas kemunculan nilai fitur pada kelas tertentu.

Karena data baru memiliki nilai fitur:
- **Harga** = Sedang
- **Diskon** = Ada
- **Ulasan** = Baik

Terdapat probabilitas bernilai 0 (misalnya `P(Harga=Sedang | Tidak Tertarik)` = 0/3). Agar prediksi stabil dan probabilitas tidak menjadi nol total saat dikalikan, kita terapkan **Laplace Smoothing (additive smoothing)** dengan $\alpha = 1$.

**Rumus Laplace Smoothing**: 
$$P(Fitur | Kelas) = \frac{\text{Frekuensi fitur di kelas} + 1}{\text{Total data kelas} + (1 \times \text{Jumlah nilai unik dari fitur tersebut})}$$

In [ ]:
# Jumlah nilai unik pada setiap fitur untuk V (Vocabulary size)
v_harga = len(set(d['Harga'] for d in data_latih))   # Murah, Sedang, Mahal (3)
v_diskon = len(set(d['Diskon'] for d in data_latih)) # Ada, Tidak Ada (2)
v_ulasan = len(set(d['Ulasan'] for d in data_latih)) # Baik, Buruk (2)

# Fungsi untuk menghitung frekuensi fitur dalam suatu kelas
def hitung_frekuensi(fitur, nilai_fitur, target_kelas):
    return sum(1 for d in data_latih if d[fitur] == nilai_fitur and d['Hasil'] == target_kelas)

# Fungsi untuk menghitung Probabilitas Likelihood dengan Laplace Smoothing
def likelihood_laplace(fitur, nilai_fitur, target_kelas, jumlah_kelas, v_unik):
    frekuensi = hitung_frekuensi(fitur, nilai_fitur, target_kelas)
    probabilitas = (frekuensi + 1) / (jumlah_kelas + v_unik)
    return frekuensi, probabilitas

# === LIKELIHOOD KELAS TERTARIK ===
print("=== Likelihood Kelas 'Tertarik' ===")
freq_h_t, prob_h_t = likelihood_laplace('Harga', 'Sedang', 'Tertarik', jumlah_tertarik, v_harga)
freq_d_t, prob_d_t = likelihood_laplace('Diskon', 'Ada', 'Tertarik', jumlah_tertarik, v_diskon)
freq_u_t, prob_u_t = likelihood_laplace('Ulasan', 'Baik', 'Tertarik', jumlah_tertarik, v_ulasan)
print(f"P(Harga=Sedang | Tertarik) = ({freq_h_t} + 1) / ({jumlah_tertarik} + {v_harga}) = {prob_h_t:.4f}")
print(f"P(Diskon=Ada | Tertarik) = ({freq_d_t} + 1) / ({jumlah_tertarik} + {v_diskon}) = {prob_d_t:.4f}")
print(f"P(Ulasan=Baik | Tertarik) = ({freq_u_t} + 1) / ({jumlah_tertarik} + {v_ulasan}) = {prob_u_t:.4f}")

# === LIKELIHOOD KELAS TIDAK TERTARIK ===
print("\n=== Likelihood Kelas 'Tidak Tertarik' ===")
freq_h_tt, prob_h_tt = likelihood_laplace('Harga', 'Sedang', 'Tidak Tertarik', jumlah_tidak_tertarik, v_harga)
freq_d_tt, prob_d_tt = likelihood_laplace('Diskon', 'Ada', 'Tidak Tertarik', jumlah_tidak_tertarik, v_diskon)
freq_u_tt, prob_u_tt = likelihood_laplace('Ulasan', 'Baik', 'Tidak Tertarik', jumlah_tidak_tertarik, v_ulasan)
print(f"P(Harga=Sedang | Tidak Tertarik) = ({freq_h_tt} + 1) / ({jumlah_tidak_tertarik} + {v_harga}) = {prob_h_tt:.4f}")
print(f"P(Diskon=Ada | Tidak Tertarik) = ({freq_d_tt} + 1) / ({jumlah_tidak_tertarik} + {v_diskon}) = {prob_d_tt:.4f}")
print(f"P(Ulasan=Baik | Tidak Tertarik) = ({freq_u_tt} + 1) / ({jumlah_tidak_tertarik} + {v_ulasan}) = {prob_u_tt:.4f}")

## Langkah 4: Menghitung Posterior
Nilai Posterior (tanpa normalisasi) dicari dengan mengalikan **Prior** kelas dengan semua **Likelihood** fiturnya.

$$Posterior = P(Kelas) \times P(Fitur_1 | Kelas) \times P(Fitur_2 | Kelas) \times P(Fitur_3 | Kelas)$$

In [ ]:
# Menghitung Posterior Kelas Tertarik
posterior_tertarik = prior_tertarik * prob_h_t * prob_d_t * prob_u_t
print(f"Posterior(Tertarik) = {prior_tertarik:.4f} * {prob_h_t:.4f} * {prob_d_t:.4f} * {prob_u_t:.4f}")
print(f"Posterior(Tertarik) = {posterior_tertarik:.6f}")

print() # baris kosong

# Menghitung Posterior Kelas Tidak Tertarik
posterior_tidak_tertarik = prior_tidak_tertarik * prob_h_tt * prob_d_tt * prob_u_tt
print(f"Posterior(Tidak Tertarik) = {prior_tidak_tertarik:.4f} * {prob_h_tt:.4f} * {prob_d_tt:.4f} * {prob_u_tt:.4f}")
print(f"Posterior(Tidak Tertarik) = {posterior_tidak_tertarik:.6f}")

## Langkah 5: Kesimpulan
Kita membandingkan kedua nilai Posterior yang didapatkan. Kelas dengan Probabilitas Posterior lebih besar merupakan hasil prediksi dari model Naive Bayes.

In [ ]:
print("=== KESIMPULAN PREDIKSI ===")
if posterior_tertarik > posterior_tidak_tertarik:
    prediksi_final = "Tertarik"
    print("Karena Posterior(Tertarik) > Posterior(Tidak Tertarik), maka prediksi akhirnya adalah kelas: TERTARIK")
elif posterior_tidak_tertarik > posterior_tertarik:
    prediksi_final = "Tidak Tertarik"
    print("Karena Posterior(Tidak Tertarik) > Posterior(Tertarik), maka prediksi akhirnya adalah kelas: TIDAK TERTARIK")
else:
    prediksi_final = "Seimbang (Tidak pasti)"
    print("Kedua Posterior bernilai sama secara probabilitas.")

print("\nHasil Klasifikasi pada Data Baru -> Kategori:", prediksi_final)

## Penjelasan Analisis

Berdasarkan hasil perhitungan probabilitas Naive Bayes dengan penerapan **Laplace Smoothing**, nilai **Posterior untuk kelas 'Tertarik'** lebih besar dibandingkan dengan nilai **Posterior untuk kelas 'Tidak Tertarik'**.

1. Kombinasi fitur **Harga = Sedang**, **Diskon = Ada**, dan **Ulasan = Baik** sangat condong mengarah pada kelas 'Tertarik' karena frekuensi pola ini pada data latih cukup dominan. 
2. Hal paling krusial terjadi pada fitur **Harga = Sedang** pada kelas **Tidak Tertarik** di mana historis data aslinya adalah 0 kasus dari 3 kasus. Jika kita tidak menggunakan *Laplace Smoothing*, proses perkalian probabilitas dari *likelihood* akan langsung menghasilkan angka posterior absolut 0. Hal ini dihindari karena ada probabilitas bahwa kejadian ini tetap dapat terjadi di masa depan meskipun sangat kecil probabilitasnya (*Zero Probability Problem*).
3. Setelah **Laplace Smoothing ($\alpha=1$)** diaplikasikan secara adil ke semua perhitungan, kedua nilai posterior dapat dibandingkan dengan akurat. Dominasi probabilitas fitur pendukung pada kelas 'Tertarik' tetap lebih kuat, sehingga klasifikasi akhir untuk pembeli dengan setelan *Harga Sedang, Ada Diskon, Ulasan Baik*, dinyatakan: **Tertarik**.